In [ ]:
%spark.pyspark

df = spark.read.option("multiline", "true").json("s3a://vdi-hse-stud/simple.json")
df.show(1, truncate=False)

In [ ]:
%spark.pyspark
from pyspark.sql.functions import col, explode

items_df = df.select(
    col("order_id"),
    col("status").alias("order_status"),
    col("payment.method").alias("payment_method"),
    col("payment.status").alias("payment_status"),
    col("customer.id").alias("customer_id"),
    col("customer.address.city").alias("city"),
    explode(col("items")).alias("item")
).select(
    "*",
    col("item.sku"),
    col("item.name").alias("item_name"),
    col("item.qty"),
    col("item.price"),
    (col("item.qty") * col("item.price")).alias("line_total")
).drop("item")

items_df.cache()
items_df.show(1,truncate=False)


In [ ]:
%spark.pyspark

items_df.coalesce(1).write.mode("overwrite").parquet("s3a://vdi-hse-stud/simple_data_exploded")

In [ ]:
%spark.pyspark
spark.read.parquet("s3a://vdi-hse-stud/simple_data_exploded").show(2,truncate=False)